In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers rank_bm25

from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi
import pandas as pd

In [ ]:
documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules.

Deep Learning takes Machine Learning further by using neural networks with many layers to
automatically discover complex patterns in data.

Retrieval-Augmented Generation, or RAG, combines a search system with a language model so answers
are grounded in real retrieved documents instead of only the model's memorized training data.
""",

    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching, allowing
systems to find results that match the meaning of a query, not just its exact words.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
""",

    "product_catalog.txt": """Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.

The standing desk converter is listed under product code SKU-7788 and ships within three business
days.

Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
""",

    "new_product_catalog.txt": """New Product Catalog

The smart fitness watch is listed under product code SKU-9911 and provides real-time health
monitoring, activity tracking, heart-rate information, and sleep analysis.

The product is available through the official product store and includes a one-year warranty.
"""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="rag_hybrid_demo",
    embedding_function=hf_embedder
)

all_chunks = []
all_chunk_ids = []
all_chunk_meta = []

for source, text in documents.items():

    doc_chunks = splitter.split_text(text)

    doc_ids = [
        f"{source}_chunk_{i}"
        for i in range(len(doc_chunks))
    ]

    doc_meta = [
        {
            "source": source,
            "chunk_index": i
        }
        for i in range(len(doc_chunks))
    ]

    collection.add(
        documents=doc_chunks,
        metadatas=doc_meta,
        ids=doc_ids
    )

    all_chunks.extend(doc_chunks)
    all_chunk_ids.extend(doc_ids)
    all_chunk_meta.extend(doc_meta)

print("Total chunks stored:", collection.count())

for cid, chunk in zip(all_chunk_ids, all_chunks):
    print(f"  {cid}: {chunk[:60]}...")

In [ ]:
def semantic_search(query, n_results=3):

    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    output = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):

        output.append({
            "text": doc,
            "source": meta["source"],
            "similarity": round(1 - distance, 4)
        })

    return output


results = semantic_search("SKU-4521")

for r in results:
    print(
        f"[{r['similarity']}] "
        f"({r['source']}) {r['text']}"
    )


# BM25

def tokenize(text):
    return text.lower().replace("-", " ").split()


tokenized_chunks = [
    tokenize(chunk)
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_chunks)


def bm25_search(query, n_results=3):

    scores = bm25.get_scores(
        tokenize(query)
    )

    ranked_idx = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:n_results]

    output = []

    for idx in ranked_idx:

        if scores[idx] > 0:

            output.append({
                "text": all_chunks[idx],
                "source": all_chunk_meta[idx]["source"],
                "bm25_score": round(
                    float(scores[idx]),
                    4
                )
            })

    return output


results = bm25_search("SKU-4521")

for r in results:
    print(
        f"[{r['bm25_score']}] "
        f"({r['source']}) {r['text']}"
    )

In [ ]:


results = bm25_search(
    "a computer system that learns from experience"
)

print("BM25 results for a meaning-based query:")

if not results:
    print("  (no keyword matches found)")

for r in results:
    print(
        f"[{r['bm25_score']}] "
        f"({r['source']}) {r['text']}"
    )




def hybrid_search(query, n_results=3, k=10):

    semantic_results = semantic_search(
        query,
        n_results=10
    )

    keyword_results = bm25_search(
        query,
        n_results=10
    )

    rrf_scores = {}
    chunk_lookup = {}

    for rank, r in enumerate(semantic_results):

        key = r["text"]

        rrf_scores[key] = (
            rrf_scores.get(key, 0)
            + 1 / (k + rank + 1)
        )

        chunk_lookup[key] = r

    for rank, r in enumerate(keyword_results):

        key = r["text"]

        rrf_scores[key] = (
            rrf_scores.get(key, 0)
            + 1 / (k + rank + 1)
        )

        chunk_lookup.setdefault(key, r)

    ranked = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:n_results]

    output = []

    for text, score in ranked:

        chunk_info = chunk_lookup[text]

        output.append({
            "text": text,
            "source": chunk_info["source"],
            "rrf_score": round(score, 5)
        })

    return output


results = hybrid_search(
    "SKU-4521 product details",
    k=10
)

for r in results:
    print(
        f"[{r['rrf_score']}] "
        f"({r['source']}) {r['text']}"
    )

In [ ]:
test_query = "What is SKU-4521?"

semantic_top = semantic_search(
    test_query,
    n_results=1
)

keyword_top = bm25_search(
    test_query,
    n_results=1
)


hybrid_top = hybrid_search(
    test_query,
    n_results=1,
    k=10
)

comparison = pd.DataFrame({
    "Method": [
        "Semantic only",
        "Keyword (BM25) only",
        "Hybrid (RRF k=10)"
    ],

    "Top Result": [
        semantic_top[0]["text"][:70] + "..."
        if semantic_top
        else "(no result)",

        keyword_top[0]["text"][:70] + "..."
        if keyword_top
        else "(no result)",

        hybrid_top[0]["text"][:70] + "..."
        if hybrid_top
        else "(no result)"
    ]
})

comparison

In [ ]:
def distance_to_similarity(distance):
    return 1 - distance


def retrieve_with_threshold(
    query,
    n_results=5,
    similarity_threshold=0.3
):

    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    filtered = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):

        similarity = distance_to_similarity(distance)

        if similarity >= similarity_threshold:

            filtered.append({
                "text": doc,
                "source": meta["source"],
                "similarity": round(
                    similarity,
                    4
                )
            })

    return filtered


empty_test = retrieve_with_threshold(
    "What's the best recipe for chocolate cake?"
)

print(
    "Chunks passing the threshold:",
    len(empty_test)
)

for r in empty_test:
    print(r)



STOPWORDS = {
    "a", "an", "the",
    "is", "are", "was", "were",
    "what", "which", "who",
    "where", "when", "why",
    "how", "can", "could",
    "would", "should",
    "tell", "me", "about",
    "please", "for", "to",
    "of", "in", "on",
    "and", "or", "with",
    "from"
}


def remove_stopwords(query):

    words = query.lower().split()

    return " ".join(
        word
        for word in words
        if word not in STOPWORDS
    )


KNOWLEDGE_BASE_TOPICS = sorted({
    meta["source"]
    for meta in all_chunk_meta
})


def smart_retrieve(
    query,
    n_results=4,
    similarity_threshold=0.3
):

    primary = retrieve_with_threshold(
        query,
        n_results=n_results,
        similarity_threshold=similarity_threshold
    )

    if primary:

        return {
            "status": "ok",
            "method": "semantic",
            "chunks": primary
        }


    hybrid_results = hybrid_search(
        query,
        n_results=n_results,
        k=10
    )

    if hybrid_results:

        return {
            "status": "ok",
            "method": "hybrid_fallback",
            "chunks": hybrid_results
        }


    cleaned_query = remove_stopwords(query)

    if cleaned_query != query:

        stopword_results = retrieve_with_threshold(
            cleaned_query,
            n_results=n_results,
            similarity_threshold=similarity_threshold
        )

        if stopword_results:

            return {
                "status": "ok",
                "method": "stopword_fallback",
                "chunks": stopword_results
            }


    return {
        "status": "empty",
        "method": None,
        "chunks": [],
        "message": (
            "I don't know based on the provided documents. "
            f"This knowledge base currently covers: "
            f"{', '.join(KNOWLEDGE_BASE_TOPICS)}."
        )
    }

In [ ]:

result = smart_retrieve(
    "What is deep learning?"
)

print(
    "Test 1 ->",
    result["status"],
    "|",
    result["method"]
)



result = smart_retrieve(
    "Tell me about SKU-4521"
)

print(
    "Test 2 ->",
    result["status"],
    "|",
    result["method"]
)

result = smart_retrieve(
    "Tell me about SKU-9911"
)

print(
    "Test 3 ->",
    result["status"],
    "|",
    result["method"]
)

for r in result["chunks"]:
    print("\nSource:", r["source"])
    print("Text:", r["text"])


query = "Can you tell me about the SKU-9911 product?"

result = smart_retrieve(query)

print("\nStopword Test")
print("Query:", query)
print("Status:", result["status"])
print("Method:", result["method"])


result = smart_retrieve(
    "What's the best recipe for chocolate cake?"
)

print(
    "\nTest 5 ->",
    result["status"],
    "|",
    result["method"]
)

if result["status"] == "empty":
    print(result["message"])



def format_context_for_llm(query, chunks):

    context_text = "\n\n".join(
        f"[Source: {c['source']}]\n{c['text']}"
        for c in chunks
    )

    return (
        f"Context:\n{context_text}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )


def safe_rag_prompt(query):

    retrieval = smart_retrieve(query)

    if retrieval["status"] == "empty":
        return retrieval["message"]

    prompt = format_context_for_llm(
        query,
        retrieval["chunks"]
    )

    return (
        f"[Retrieved via "
        f"'{retrieval['method']}']\n\n"
        f"{prompt}"
    )


print(
    "\n",
    safe_rag_prompt("What is SKU-9911?")
)

print("\n" + "=" * 70 + "\n")

print(
    safe_rag_prompt(
        "What's the best recipe for chocolate cake?"
    )
)